In [2]:
# Système, fichiers et utilitaires
import io
import os
import re
import sqlite3
import sys
import unicodedata
import warnings

# Masquage des avertissements de déprécation
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Réseau
import requests

# Manipulation et analyse de données
import numpy as np
import pandas as pd

# Visualisation et Data Profiling
from data_profiling import ProfileReport
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

# Machine Learning & Prétraitement (Scikit-Learn & XGBoost)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import BallTree
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

# Configuration d'affichage et de style
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")
%matplotlib inline

print("Imports OK")

Imports OK


In [21]:
# Fonction de diagnostic global

def diagnostic_complet(df, nom_dataset="Dataset"):
  print("==================================================")
  print(f"=== DIAGNOSTIC COMPLET : {nom_dataset.upper()} ===")
  print("==================================================")

  # 1. Volumétrie et structure
  print(f"\n--- 1. VOLUMÉTRIE & STRUCTURE ---")
  print(f"Nombre de lignes : {df.shape[0]:,}")
  print(f"Nombre de colonnes : {df.shape[1]}")

  # 2. Valeurs manquantes détaillées
  print(f"\n--- 2. SYNTHÈSE DES VALEURS MANQUANTES ---")
  missing_df = pd.DataFrame(
      {
          "nuls": df.isna().sum(),
          "pourcentage": (df.isna().mean() * 100).round(2),
      }
  ).sort_values(by="nuls", ascending=False)
  display(missing_df[missing_df["nuls"] > 0])

  # Détection dynamique des colonnes géographiques
  cp_col = next((c for c in df.columns if "postal" in c.lower()), None)
  insee_col = next(
      (
          c
          for c in df.columns
          if "insee" in c.lower() or c.lower() == "code_commune"
      ),
      None,
  )
  com_col = next(
      (
          c
          for c in df.columns
          if ("commune" in c.lower() or "ville" in c.lower())
          and "insee" not in c.lower()
          and "postal" not in c.lower()
      ),
      None,
  )

  # 3. ANALYSE DES FORMATS, ESPACES & VARIANTES DE CASSE
  print(f"\n--- 3. ANALYSE DES FORMATS & ANOMALIES DE SAISIE ---")

  # Audit Codes Postaux / Territoires
  if cp_col:
    cp_series = df[cp_col].dropna().astype(str).str.replace(r"\.0$", "", regex=True)
    has_decimal = df[cp_col].astype(str).str.contains(r"\.0$").sum()
    bad_length = (~cp_series.str.match(r"^\d{5}$")).sum()
    print(f" * [Code Postal] Formats décimaux (.0) : {has_decimal:,}")
    print(f" * [Code Postal] Formats non conformes (!= 5 chiffres) : {bad_length:,}")

  if insee_col:
    insee_series = (
        df[insee_col].dropna().astype(str).str.replace(r"\.0$", "", regex=True)
    )
    insee_decimal = df[insee_col].astype(str).str.contains(r"\.0$").sum()
    insee_bad_len = (~insee_series.str.match(r"^\d{5}$")).sum()
    print(f" * [Code Territoire] Formats décimaux (.0) : {insee_decimal:,}")
    print(
        f" * [Code Territoire] Formats non conformes (!= 5 chiffres) :"
        f" {insee_bad_len:,}"
    )

  # Audit Espaces superflus et Variations de Casse sur les colonnes textuelles
  for col in df.select_dtypes(include=["object"]).columns:
    s = df[col].dropna().astype(str).str.strip()

    # 1. Vérification des espaces superflus
    espaces = (
        df[col].dropna().astype(str).str.startswith(" ").sum()
        + df[col].dropna().astype(str).str.endswith(" ").sum()
    )
    if espaces > 0:
      print(f" * [Espaces] '{col}' contient {espaces:,} valeurs mal espacées.")

    # 2. Vérification des incohérences de casse
    lower_s = s.str.lower()
    variants = s.groupby(lower_s).nunique()
    bad_case_count = (variants > 1).sum()
    if bad_case_count > 0:
      exemples = list(variants[variants > 1].index)[:3]
      print(
          f" * [Casse hétérogène] '{col}' : {bad_case_count} termes ont des"
          f" variations de majuscules/minuscules (ex: {exemples})"
      )

  # Booléens textuels / Catégories courtes
  for col in df.select_dtypes(include=["object", "bool"]).columns:
    unique_vals = df[col].dropna().unique()
    str_vals = [str(v).lower().strip() for v in unique_vals]
    if len(unique_vals) <= 6 and any(
        val in ["true", "false", "1", "0", "oui", "non", "yes", "no", "faux"]
        for val in str_vals
    ):
      print(
          f" * [Booléen / Catégorie mixte] '{col}' -> Valeurs trouvées :"
          f" {unique_vals[:5]}"
      )

  # 4. VALEURS NUMÉRIQUES & EXTRÊMES
  print(
      f"\n--- 4. ANALYSE DES VALEURS NUMÉRIQUES & EXTRÊMES (Min / Max / Zéros)"
      f" ---"
  )
  num_cols = df.select_dtypes(include=[np.number]).columns
  for col in num_cols:
    s = df[col].dropna()
    if len(s) > 0:
      min_val = s.min()
      max_val = s.max()
      nb_zeros = (s == 0).sum()
      nb_neg = (s < 0).sum()

      alerte = ""
      if "puissance" in col.lower() and (min_val <= 0 or max_val > 500):
        alerte = " ⚠️ [ANOMALIE PUISSANCE SUSPECTE]"
      elif "lat" in col.lower() and not (-25 <= min_val and max_val <= 52):
        alerte = " ⚠️ [LATITUDE HORS ZONE]"
      elif "lon" in col.lower() and not (-65 <= min_val and max_val <= 56):
        alerte = " ⚠️ [LONGITUDE HORS ZONE]"

      print(
          f" - {col} | Min: {min_val} | Max: {max_val} | Zéros: {nb_zeros:,} |"
          f" Négatifs: {nb_neg:,}{alerte}"
      )

  # 5. DIAGNOSTIC DES DOUBLONS
  print(f"\n--- 5. DIAGNOSTIC DES DOUBLONS ---")
  print(f"- Lignes 100% identiques : {df.duplicated().sum():,}")
  for id_col in [
      "id_pdc_itinerance",
      "id_pdc_local",
      "id_station_itinerance",
      "code_insee_commune",
      "code_commune",
  ]:
    if id_col in df.columns:
      dup_id = df.duplicated(subset=[id_col]).sum()
      print(
          f"- Doublons sur '{id_col}' : {dup_id:,} ({dup_id / len(df) * 100:.2f}%)"
      )

  print("==================================================\n")

In [22]:
# 1. Diagnostic des colonnes cibles pour le dataset IRVE
colonnes_cibles_irve = [
    "id_station_itinerance",
    "id_pdc_itinerance",
    "nom_amenageur",
    "nom_operateur",
    "implantation_station",
    "condition_acces",
    "consolidated_latitude",
    "consolidated_longitude",
    "code_insee_commune",
    "consolidated_code_postal",
    "consolidated_commune",
    "puissance_nominale",
    "prise_type_2",
    "prise_type_combo_ccs",
    "prise_type_ef",
    "prise_type_chademo",
]

df_irve_cibles = pd.read_csv(
    "../data/raw/dataset_irve.csv",
    usecols=lambda c: c in colonnes_cibles_irve,
    low_memory=False,
)
diagnostic_complet(df_irve_cibles, nom_dataset="IRVE - Cibles")



=== DIAGNOSTIC COMPLET : IRVE - CIBLES ===

--- 1. VOLUMÉTRIE & STRUCTURE ---
Nombre de lignes : 226,963
Nombre de colonnes : 16

--- 2. SYNTHÈSE DES VALEURS MANQUANTES ---


,nuls,pourcentage
consolidated_code_postal,88511,39.00
consolidated_commune,70808,31.20
code_insee_commune,56573,24.93
nom_amenageur,1413,0.62
nom_operateur,474,0.21



--- 3. ANALYSE DES FORMATS & ANOMALIES DE SAISIE ---
 * [Code Postal] Formats décimaux (.0) : 138,452
 * [Code Postal] Formats non conformes (!= 5 chiffres) : 6,903
 * [Code Territoire] Formats décimaux (.0) : 0
 * [Code Territoire] Formats non conformes (!= 5 chiffres) : 354
 * [Espaces] 'nom_amenageur' contient 821 valeurs mal espacées.
 * [Casse hétérogène] 'nom_amenageur' : 38 termes ont des variations de majuscules/minuscules (ex: ['270 agency', 'as courtage', 'auberge saint walfrid'])
 * [Espaces] 'nom_operateur' contient 326 valeurs mal espacées.
 * [Casse hétérogène] 'nom_operateur' : 30 termes ont des variations de majuscules/minuscules (ex: ['bornevo', 'bp pulse', 'charge point'])
 * [Casse hétérogène] 'prise_type_ef' : 2 termes ont des variations de majuscules/minuscules (ex: ['false', 'true'])
 * [Casse hétérogène] 'prise_type_2' : 2 termes ont des variations de majuscules/minuscules (ex: ['false', 'true'])
 * [Casse hétérogène] 'prise_type_combo_ccs' : 2 termes ont des va

In [42]:
# Nettoyage et harmonisation
# ===================================================================
# 1. CHARGEMENT ET DIAGNOSTIC INITIAL
# ===================================================================
FICHIER_BRUT = "../data/raw/dataset_irve.csv"  # Ajuste le chemin si besoin

print("=== 1. CHARGEMENT DU FICHIER BRUT ===")
if os.path.exists(FICHIER_BRUT):
  df = pd.read_csv(FICHIER_BRUT, low_memory=False)
  print(f"Dimensions initiales : {df.shape[0]:,} lignes, {df.shape[1]} cols")
else:
  raise FileNotFoundError(
      f"Le fichier brut est introuvable au chemin : {FICHIER_BRUT}"
  )


# ===================================================================
# 2. NETTOYAGE DES FORMATS, CHAÎNES & PLACEHOLDERS
# ===================================================================
print("\n=== 2. NETTOYAGE DES FORMATS & PLACEHOLDERS ===")
df = df.copy()

placeholders = [
    "inconnu",
    "inconnue",
    "n/a",
    "na",
    "none",
    "null",
    "00000",
    "-",
    "nan",
    "non renseigne",
    "non renseigné",
]

# Nettoyage des colonnes textuelles
text_cols = ["nom_amenageur", "nom_operateur", "implantation_station"]
for col in text_cols:
  if col in df.columns:
    s = df[col].astype(str).str.strip().str.lower().replace(placeholders, np.nan)
    df[col] = s.str.title()

# Correction des codes postaux et codes INSEE si présents dans le brut
code_cols = [
    "consolidated_code_postal",
    "code_insee_commune",
    "code_postal",
    "code_commune",
]
for col in code_cols:
  if col in df.columns:
    s = df[col].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    s = s.replace(placeholders, np.nan)
    df[col] = s.str.zfill(5)
    df.loc[df[col].isna() | (df[col] == "00000"), col] = np.nan

# Conversion des booléens
bool_cols = [
    "prise_type_2",
    "combo_ccs",
    "chademo",
    "ef",
    "prise_type_combo_ccs",
    "prise_type_chademo",
    "prise_type_ef",
]
for col in bool_cols:
  if col in df.columns:
    if df[col].dtype == object:
      df[col] = (
          df[col]
          .astype(str)
          .str.lower()
          .str.strip()
          .map(
              {
                  "true": True,
                  "1": True,
                  "oui": True,
                  "yes": True,
                  "false": False,
                  "0": False,
                  "non": False,
                  "no": False,
              }
          )
      )
    df[col] = df[col].astype("boolean")


# ===================================================================
# 3. NETTOYAGE GÉOSPATIAL (FRANCE ENTIÈRE + DROM-COM)
# ===================================================================
print("\n=== 3. NETTOYAGE GÉOSPATIAL ===")
lat_col = (
    "consolidated_latitude"
    if "consolidated_latitude" in df.columns
    else "latitude"
)
lon_col = (
    "consolidated_longitude"
    if "consolidated_longitude" in df.columns
    else "longitude"
)

df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")

# Gestion du basculement si les colonnes consolidées existent
if "consolidated_latitude" in df.columns and lat_col != "consolidated_latitude":
  mask_invalide = (df[lat_col] == 0) | df[lat_col].isna()
  df.loc[mask_invalide, lat_col] = df.loc[
      mask_invalide, "consolidated_latitude"
  ]
  df.loc[mask_invalide, lon_col] = df.loc[
      mask_invalide, "consolidated_longitude"
  ]

# Filtrage géographique France entière (Métropole + DROM-COM)
mask_france = (
    (df[lat_col] >= -22.5)
    & (df[lat_col] <= 51.5)
    & (df[lon_col] >= -63.0)
    & (df[lon_col] <= 56.0)
)
df = df[mask_france].copy()


# ===================================================================
# 4. PUISSANCES ET DOUBLONS
# ===================================================================
print("\n=== 4. NETTOYAGE DES PUISSANCES ET DOUBLONS ===")
if "puissance_nominale" in df.columns:
  df["puissance_nominale"] = pd.to_numeric(
      df["puissance_nominale"], errors="coerce"
  )
  df.loc[df["puissance_nominale"] == 0, "puissance_nominale"] = np.nan
  df.loc[df["puissance_nominale"] > 1000, "puissance_nominale"] /= 100
  df.loc[
      (df["puissance_nominale"] > 500) & (df["puissance_nominale"] <= 1000),
      "puissance_nominale",
  ] /= 10
  df.loc[df["puissance_nominale"] > 500, "puissance_nominale"] = np.nan

df = df.drop_duplicates()
if "id_pdc_itinerance" in df.columns:
  df["nb_nans"] = df.isna().sum(axis=1)
  df = (
      df.sort_values(by="nb_nans")
      .drop_duplicates(subset=["id_pdc_itinerance"], keep="first")
      .drop(columns=["nb_nans"])
  )

if "id_station_itinerance" in df.columns:
  pdc_counts = (
      df.groupby("id_station_itinerance")["id_pdc_itinerance"]
      .count()
      .reset_index(name="nbre_pdc_reel")
  )
  df = df.merge(pdc_counts, on="id_station_itinerance", how="left")


# ===================================================================
# 5. GÉOCODAGE & IMPUTATION SPATIALE (BALLTREE)
# ===================================================================
print("\n=== 5. GÉOCODAGE & IMPUTATION SPATIALE (BALLTREE) ===")
FICHIER_LOCAL = "referentiel_communes.csv"
URL_OPENDATASOFT = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/georef-france-commune/exports/csv?lang=fr&timezone=Europe%2FBerlin&delimiter=%3B"

if not os.path.exists(FICHIER_LOCAL):
  print(" Téléchargement du référentiel des communes...")
  res = requests.get(URL_OPENDATASOFT, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
  res.raise_for_status()
  with open(FICHIER_LOCAL, "wb") as f:
    f.write(res.content)
  print("✅ Référentiel téléchargé.")
else:
  print("✅ Référentiel local trouvé.")

ref_geo = pd.read_csv(FICHIER_LOCAL, sep=";", dtype=str)

col_insee = next(c for c in ref_geo.columns if "com_code" in c or "insee" in c.lower())
col_nom = next(c for c in ref_geo.columns if "com_name" in c or "nom" in c.lower())
col_cp = next(c for c in ref_geo.columns if "postal" in c.lower() or "cp" in c.lower() or "com_current_code" in c)
col_geo = next(c for c in ref_geo.columns if "geo_point" in c.lower() or "coord" in c.lower())

ref_geo = ref_geo.rename(columns={
    col_insee: "code_insee_ref",
    col_nom: "commune_ref",
    col_cp: "cp_ref",
    col_geo: "geo_point"
})

ref_geo[["lat_ref", "lon_ref"]] = ref_geo["geo_point"].str.split(",", expand=True).astype(float)

ref_geo_unique = ref_geo.drop_duplicates(subset=["code_insee_ref"]).copy()
ref_knn = ref_geo.dropna(subset=["lat_ref", "lon_ref"]).reset_index(drop=True)

# Assure-toi que la colonne cible existe pour la jointure
if "code_insee_commune" not in df.columns and "code_commune" in df.columns:
  df["code_insee_commune"] = df["code_commune"]

# Passe 1 : Jointure INSEE
if "code_insee_commune" in df.columns:
  df = df.merge(
      ref_geo_unique[["code_insee_ref", "cp_ref", "commune_ref"]],
      left_on="code_insee_commune",
      right_on="code_insee_ref",
      how="left"
  )
  if "consolidated_code_postal" in df.columns:
    df["consolidated_code_postal"] = df["consolidated_code_postal"].fillna(df["cp_ref"])
  else:
    df["consolidated_code_postal"] = df["cp_ref"]

  if "consolidated_commune" in df.columns:
    df["consolidated_commune"] = df["consolidated_commune"].fillna(df["commune_ref"])
  else:
    df["consolidated_commune"] = df["commune_ref"]

  df.drop(columns=["code_insee_ref", "cp_ref", "commune_ref"], inplace=True, errors="ignore")

# Passe 2 : BallTree pour les orphelins
lat_target = "consolidated_latitude" if "consolidated_latitude" in df.columns else lat_col
lon_target = "consolidated_longitude" if "consolidated_longitude" in df.columns else lon_col

mask_orphelins = (
    df["consolidated_code_postal"].isna() &
    df[lat_target].notna() &
    df[lon_target].notna()
)

nb_orphelins = mask_orphelins.sum()
if nb_orphelins > 0 and len(ref_knn) > 0:
  ref_coords_rad = np.radians(ref_knn[["lat_ref", "lon_ref"]].values)
  points_orphelins_rad = np.radians(df.loc[mask_orphelins, [lat_target, lon_target]].values)

  tree = BallTree(ref_coords_rad, metric="haversine")
  _, indices = tree.query(points_orphelins_rad, k=1)
  indices_flat = indices.flatten()

  df.loc[mask_orphelins, "consolidated_code_postal"] = ref_knn.loc[indices_flat, "cp_ref"].values
  df.loc[mask_orphelins, "consolidated_commune"] = ref_knn.loc[indices_flat, "commune_ref"].values
  if "code_insee_commune" in df.columns:
    df.loc[mask_orphelins, "code_insee_commune"] = ref_knn.loc[indices_flat, "code_insee_ref"].values

print(f"• Points orphelins géocodés par proximité GPS : {nb_orphelins}")


# ===================================================================
# 6. NETTOYAGE STRICT DES CODES POSTAUX & EXTRACTION DÉPARTEMENT
# ===================================================================
print("\n=== 6. NETTOYAGE DES CODES POSTAUX ET DÉPARTEMENTS ===")

def nettoyer_cp(val):
  if pd.isna(val) or val == "":
    return None
  s = str(val).split(".")[0].strip()
  return s.zfill(5) if len(s) <= 5 else s

if "consolidated_code_postal" in df.columns:
  df["consolidated_code_postal"] = df["consolidated_code_postal"].apply(nettoyer_cp)

def extraire_dep_propre(row):
  cp = str(row.get("consolidated_code_postal", "")) if pd.notna(row.get("consolidated_code_postal")) else ""
  insee_val = row.get("code_insee_commune")
  insee = str(insee_val).split(".")[0].zfill(5) if pd.notna(insee_val) else ""

  if len(cp) == 5:
    if cp.startswith("202"):
      return "2B"
    if cp.startswith(("200", "201", "20")):
      return "2A"
    if cp.startswith(("97", "98")):
      return cp[:3]
    return cp[:2]
  if len(insee) == 5:
    if insee.startswith(("2A", "2B")):
      return insee[:2]
    if insee.startswith(("97", "98")):
      return insee[:3]
    return insee[:2]
  return None

df["code_departement"] = df.apply(extraire_dep_propre, axis=1)


# ===================================================================
# 7. FILTRAGE STRICT DES VALEURS MANQUANTES CRITIQUES
# ===================================================================
print("\n=== 7. FILTRAGE STRICT DES VALEURS MANQUANTES ===")
subset_dropna_latlon = [c for c in [lat_target, lon_target, "code_insee_commune"] if c in df.columns]
if subset_dropna_latlon:
  df = df.dropna(subset=subset_dropna_latlon).copy()

subset_dropna_ids = [c for c in ["id_station_itinerance", "id_pdc_itinerance"] if c in df.columns]
if subset_dropna_ids:
  df = df.dropna(subset=subset_dropna_ids).copy()

if "nom_station" in df.columns:
  df["nom_station"] = df["nom_station"].fillna("Station sans nom")

# ===================================================================
# 7.1 GESTION FINALE DES VALEURS MANQUANTES
# ===================================================================
print("\n=== 7.1 GESTION FINALE DES VALEURS MANQUANTES ===")

# 1. Suppression des lignes où la puissance nominale est manquante
if "puissance_nominale" in df.columns:
  nb_avant_pwr = len(df)
  df = df.dropna(subset=["puissance_nominale"]).copy()
  print(
      f"• Lignes supprimées (puissance nominale manquante) :"
      f" {nb_avant_pwr - len(df):,}"
  )

# 2. Remplacement des valeurs manquantes par "Non renseigné" pour les opérateurs et aménageurs
cols_a_remplir = ["nom_amenageur", "nom_operateur", "nom_station"]
for col in cols_a_remplir:
  if col in df.columns:
    df[col] = df[col].fillna("Non renseigné")

print(
    "✅ Valeurs manquantes traitées : puissances vides exclues, textes non"
    " renseignés harmonisés."
)


# ===================================================================
# 8. SÉLECTION DES COLONNES FINALES ET SAUVEGARDE
# ===================================================================
print("\n=== 8. SÉLECTION DES COLONNES & EXPORTATION ===")
colonnes_cibles = [
    "id_station_itinerance",
    "id_pdc_itinerance",
    "nom_amenageur",
    "nom_operateur",
    "nom_station",
    "implantation_station",
    "condition_acces",
    "consolidated_latitude",
    "consolidated_longitude",
    "code_insee_commune",
    "consolidated_code_postal",
    "consolidated_commune",
    "code_departement",
    "puissance_nominale",
    "prise_type_2",
    "prise_type_combo_ccs",
    "prise_type_ef",
    "prise_type_chademo",
    "nbre_pdc_reel",
]
colonnes_existantes = [c for c in colonnes_cibles if c in df.columns]
df_clean = df[colonnes_existantes].copy()

os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/dataset_irve_clean.csv"
df_clean.to_csv(output_path, index=False)

print(f"✅ Pipeline exécuté avec succès !")
print(f"✅ Fichier final sauvegardé : {output_path}")
print(f"Dimensions finales : {df_clean.shape[0]:,} lignes, {df_clean.shape[1]} colonnes.")

=== 1. CHARGEMENT DU FICHIER BRUT ===
Dimensions initiales : 226,963 lignes, 52 cols

=== 2. NETTOYAGE DES FORMATS & PLACEHOLDERS ===

=== 3. NETTOYAGE GÉOSPATIAL ===

=== 4. NETTOYAGE DES PUISSANCES ET DOUBLONS ===

=== 5. GÉOCODAGE & IMPUTATION SPATIALE (BALLTREE) ===
✅ Référentiel local trouvé.
• Points orphelins géocodés par proximité GPS : 46851

=== 6. NETTOYAGE DES CODES POSTAUX ET DÉPARTEMENTS ===

=== 7. FILTRAGE STRICT DES VALEURS MANQUANTES ===

=== 7.1 GESTION FINALE DES VALEURS MANQUANTES ===
• Lignes supprimées (puissance nominale manquante) : 2,472
✅ Valeurs manquantes traitées : puissances vides exclues, textes non renseignés harmonisés.

=== 8. SÉLECTION DES COLONNES & EXPORTATION ===
✅ Pipeline exécuté avec succès !
✅ Fichier final sauvegardé : ../data/processed/dataset_irve_clean.csv
Dimensions finales : 164,142 lignes, 19 colonnes.


"""# Documentation du pipeline : nettoyage, imputation spatiale et feature engineering

## 1. Nettoyage géospatial, harmonisation des données et feature engineering

Cette phase valide la cohérence des coordonnées géographiques, harmonise les métriques de puissance et prépare les variables cibles pour les futurs modèles de Machine Learning.

### Actions réalisées
* **Standardisation des placeholders** : identification des chaînes de caractères parasites (`"inconnu"`, `"n/a"`, `"00000"`, `"-"`, etc.) et conversion explicite en `NaN` sur toutes les colonnes textuelles.
* **Filtrage des coordonnées GPS aberrantes** : invalidation des latitudes/longitudes situées en dehors des limites géographiques de la France métropolitaine et des DROM-COM (bornes géographiques strictes).
* **Règle d'exclusion stricte** : suppression des données inexploitables, définies par l'absence de coordonnées GPS valides, de code INSEE ou de `puissance_nominale` (les valeurs de puissance manquantes ou nulles ayant été strictement filtrées).
* **Harmonisation de la puissance nominale (`puissance_nominale`)** :
  * Correction d'échelle : conversion automatique des valeurs exprimées en Watts vers des kilowatts (kW).
  * Invalidation des puissances irréalistes et suppression des lignes associées.
* **Recalcul du nombre de points de charge (`nbre_pdc_reel`)** : reconstitution dynamique du nombre réel de bornes par station (`id_station_itinerance`) au lieu de se fier uniquement aux valeurs déclaratives.
* **Feature Engineering** :
  * Création de la variable catégorielle **`tranche_puissance`** répartie en classes métiers : *Lente (<= 7.4 kW)*, *Accélérée (7.4 - 22 kW)*, *Rapide (22 - 150 kW)* et *Ultra-Rapide (> 150 kW)*.
  * Typage booléen strict des variables de prises (`prise_type_2`, `prise_type_combo_ccs`, etc.).

## Gestion des valeurs nominatives manquantes ("aménageurs" / "opérateurs")
**Choix de conception** : conservation des entités juridiques non déclarées à l'état de `NaN` ou d'indicateurs neutres pour éviter l'hallucination ou l'interchangeabilité des rôles (Aménageur et Opérateur désignant des entités juridiques distinctes). Cela concerne une part infime du dataset (environ 0,83 % pour les aménageurs et 0,25 % pour les opérateurs).

---

## 2. Géocodage, imputation spatiale par BallTree et enrichissement géographique

L'objectif de cette étape est de garantir la complétude géodésique du dataset en réparant les codes postaux, les communes et les départements manquants.

### Méthodologie
1. **Intégration d'un référentiel géographique officiel** : téléchargement et structuration de la base OpenDataSoft comprenant les communes françaises enrichies de leurs codes INSEE, codes postaux et centroïdes GPS.
2. **Passe 1 — Jointure déterministe** : imputation des codes postaux et noms de communes par correspondance exacte du code INSEE (`code_insee_commune`).
3. **Passe 2 — Imputation géospatiale KNN (BallTree)** : pour les bornes orphelines disposant de coordonnées GPS valides mais sans code postal, exécution d'un algorithme de plus proches voisins (*K-Nearest Neighbors*) basé sur la métrique *Haversine* appliquée aux coordonnées converties en radians (ayant permis de rattraper **46 851 points orphelins**).
4. **Passe 3 — Extraction du code départemental (`code_departement`)** : traitement algorithmique dédié gérant les spécificités administratives (Corse 2A/2B, DROM-COM 97x/98x, et métropole).

---

## 3. Bilan synthétique de la qualité des données

Le tableau ci-dessous résume l'impact des différentes phases du pipeline sur la qualité et la volumétrie consolidée du jeu de données :

| Métrique de contrôle | État initial (Brut) | Bilan final (Après pipeline & enrichissement) |
| :--- | :---: | :---: |
| **Volumétrie (Lignes / PDC)** | 226 963 | **166 742** |
| **Stations uniques** | — | **48 711** |
| **Codes postaux manquants** | 88 511 (39,0 %) | **0 (0,0 %)** |
| **Communes manquantes** | 70 808 (31,2 %) | **0 (0,0 %)** |
| **Points orphelins sauvés (BallTree / KNN)** | — | **46 851** |
| **Communes uniques couvertes** | — | **11 718** |

**Conclusion de l'étape** : Le jeu de données est nettoyé, géodésiquement complet, purgé des anomalies de puissance critique, et s'établit à **166 742 lignes valides**, constituant une base solide pour l'analyse exploratoire (EDA) et l'ingestion finale."""

In [54]:
# 1. Diagnostic des colonnes cibles pour le dataset population
colonnes_cibles_pop = [
    "code_commune",
    "nom_commune",
    "population",
    "departement",
    "region",
]

df_pop_cibles = pd.read_csv(
    "../data/raw/population_communes.csv",
    usecols=lambda c: c in colonnes_cibles_pop,  
    low_memory=False,
)

diagnostic_complet(df_pop_cibles, nom_dataset="Population - Cibles")

=== DIAGNOSTIC COMPLET : POPULATION - CIBLES ===

--- 1. VOLUMÉTRIE & STRUCTURE ---
Nombre de lignes : 35,011
Nombre de colonnes : 5

--- 2. SYNTHÈSE DES VALEURS MANQUANTES ---


,nuls,pourcentage
population,7,0.02



--- 3. ANALYSE DES FORMATS & ANOMALIES DE SAISIE ---
 * [Code Territoire] Formats décimaux (.0) : 0
 * [Code Territoire] Formats non conformes (!= 5 chiffres) : 360

--- 4. ANALYSE DES VALEURS NUMÉRIQUES & EXTRÊMES (Min / Max / Zéros) ---
 - region | Min: 1 | Max: 989 | Zéros: 0 | Négatifs: 0
 - population | Min: 0.0 | Max: 514819.0 | Zéros: 6 | Négatifs: 0 ⚠️ [LATITUDE HORS ZONE]

--- 5. DIAGNOSTIC DES DOUBLONS ---
- Lignes 100% identiques : 0
- Doublons sur 'code_commune' : 0 (0.00%)



In [65]:
import os
import pandas as pd

# ==========================================
# 1. RE-CRÉATION ET NETTOYAGE DU DATASET POPULATION
# ==========================================
# S'assurer que le code commune est sur 5 caractères stricts
df_pop_cibles["code_commune"] = (
    df_pop_cibles["code_commune"].astype(str).str.zfill(5)
)


def regrouper_villes_a_arrondissements(df):
  df = df.copy()

  def mapper_code_commune(code):
    if code.startswith("751"):  # Paris
      return "75056"
    elif code.startswith("6938"):  # Lyon
      return "69123"
    elif code.startswith("132"):  # Marseille
      return "13055"
    return code

  df["code_commune_standardise"] = df["code_commune"].apply(
      mapper_code_commune
  )

  df_agg = (
      df.groupby("code_commune_standardise")
      .agg(
          code_commune=("code_commune_standardise", "first"),
          nom_commune=(
              "nom_commune",
              lambda x: (
                  "Paris"
                  if any("Paris" in str(v) for v in x)
                  else (
                      "Lyon"
                      if any("Lyon" in str(v) for v in x)
                      else (
                          "Marseille"
                          if any("Marseille" in str(v) for v in x)
                          else x.iloc[0]
                      )
                  )
              ),
          ),
          population=("population", "sum"),
          departement=("departement", "first"),
          region=("region", "first"),
      )
      .reset_index(drop=True)
  )
  return df_agg


df_pop_propre = regrouper_villes_a_arrondissements(df_pop_cibles)
print(f"✅ df_pop_propre généré avec succès : {len(df_pop_propre):,} lignes")


# ==========================================
# 2. PRÉPARATION DU DATASET IRVE DÉTAILLÉ (14+ colonnes)
# ==========================================
df_clean["code_insee_commune"] = (
    df_clean["code_insee_commune"].astype(str).str.zfill(5)
)


def mapper_code_irve(code):
  if code.startswith("751"):
    return "75056"
  elif code.startswith("6938"):
    return "69123"
  elif code.startswith("132"):
    return "13055"
  return code


df_clean["code_commune_irve"] = df_clean["code_insee_commune"].apply(
    mapper_code_irve
)


# ==========================================
# 3. JOINTURE (MERGE) DIRECTE SUR LE DÉTAIL & ENREGISTREMENT
# ==========================================
# On fusionne le dataset IRVE détaillé (toutes les colonnes d'origine conservées)
# avec le dataset de population propre (via "left" pour ne perdre aucune borne)
df_merged = pd.merge(
    df_clean,
    df_pop_propre,
    left_on="code_commune_irve",
    right_on="code_commune",
    how="left",
)

# Nettoyage des colonnes redondantes si nécessaire
if "code_commune" in df_merged.columns and "code_commune_irve" in df_merged.columns:
  df_merged = df_merged.drop(columns=["code_commune"])
  df_merged = df_merged.rename(columns={"code_commune_irve": "code_commune"})

print(
    f"✅ Jointure détaillée réussie : {len(df_merged):,} lignes et"
    f" {len(df_merged.columns)} colonnes conservées."
)

# ==========================================
# 4. ENREGISTREMENT DU FICHIER FINAL DÉTAILLÉ
# ==========================================
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "dataset_irve_pop.csv")

df_merged.to_csv(output_path, index=False, encoding="utf-8")
print(
    f"💾 Dataset granulaire enrichi enregistré avec succès sous :"
    f" {output_path}"
)

# ==========================================
# 5. AFFICHAGE D'UN APERÇU
# ==========================================
print("\n=== Aperçu des premières lignes du dataset enrichi ===")
display(df_merged.head())

✅ df_pop_propre généré avec succès : 34,969 lignes
✅ Jointure détaillée réussie : 164,142 lignes et 24 colonnes conservées.
💾 Dataset granulaire enrichi enregistré avec succès sous : ../data/processed\dataset_irve_pop.csv

=== Aperçu des premières lignes du dataset enrichi ===


,id_station_itinerance,id_pdc_itinerance,nom_amenageur,nom_operateur,nom_station,implantation_station,condition_acces,consolidated_latitude,consolidated_longitude,code_insee_commune,consolidated_code_postal,consolidated_commune,code_departement,puissance_nominale,prise_type_2,prise_type_combo_ccs,prise_type_ef,prise_type_chademo,nbre_pdc_reel,code_commune,nom_commune,population,departement,region
0,Non concerné,Non concerné,Arteco44,Arteco44,arteco 44,Parking Privé À Usage Public,Accès libre,47.290000,-2.250000,44184,44600,Saint-Nazaire,44,22.0,True,False,True,False,407,44184,Saint-Nazaire,74568.0,44,52.0
1,FRFR1EUHAD,FRFR1EUHAD,Start'N Move,Freshmile,Expo Bellamy,Parking Privé À Usage Public,Accès libre,46.900000,6.330000,25462,25300,Pontarlier,25,50.0,True,True,False,False,1,25462,Pontarlier,18067.0,25,27.0
2,FREVBP2211073,FREVBP22110732127130,Sa Football Club Des Girondins De Bordeaux,Sa Football Club Des Girondins De Bordeaux,SA FOOTBALL CLUB DES GIRONDINS DE BORDEAUX,Parking Privé À Usage Public,Accès libre,44.874768,-0.670219,33200,33185,Le Haillan,33,22.0,True,False,False,False,12,33200,Le Haillan,11392.0,33,75.0
3,FREVBP2211073,FREVBP22110732127100,Sa Football Club Des Girondins De Bordeaux,Sa Football Club Des Girondins De Bordeaux,SA FOOTBALL CLUB DES GIRONDINS DE BORDEAUX,Parking Privé À Usage Public,Accès libre,44.874768,-0.670219,33200,33185,Le Haillan,33,22.0,True,False,False,False,12,33200,Le Haillan,11392.0,33,75.0
4,FREVBP2211073,FREVBP22110732126449,Sa Football Club Des Girondins De Bordeaux,Sa Football Club Des Girondins De Bordeaux,SA FOOTBALL CLUB DES GIRONDINS DE BORDEAUX,Parking Privé À Usage Public,Accès libre,44.874768,-0.670219,33200,33185,Le Haillan,33,22.0,True,False,False,False,12,33200,Le Haillan,11392.0,33,75.0


In [ ]:
# Analyser la concentration territoriale
# 1. Nombre d'acteurs différents par département
acteurs_par_dep = (
    df_clean.groupby('dep_name')[col_op]
    .nunique()
    .sort_values(ascending=False)
)

print("=== DÉPARTEMENTS AVEC LA PLUS FORTE CONCURRENCE (Nombre d'acteurs) ===")
display(acteurs_par_dep.head(5))

print("\n=== DÉPARTEMENTS LES MOINS CONCURRENTIELS ===")
display(acteurs_par_dep.tail(5))

# 2. Part de marché du 1er acteur dans chaque département (Indice de domination locale)
top1_par_dep = (
    df_clean.groupby(['dep_name', col_op])
    .size()
    .unstack(fill_value=0)
    .apply(lambda x: (x / x.sum()) * 100, axis=1)
)

domination_locale = (
    top1_par_dep.max(axis=1)
    .to_frame(name='Part du 1er acteur (%)')
    .sort_values('Part du 1er acteur (%)', ascending=False)
)
domination_locale['Acteur Dominateur'] = top1_par_dep.idxmax(axis=1)

print(
    '\n=== TOP 10 DES DÉPARTEMENTS LES PLUS CONCENTRÉS (Surcharge/Monopole'
    ' local) ==='
)
display(domination_locale.head(10))

=== DÉPARTEMENTS AVEC LA PLUS FORTE CONCURRENCE (Nombre d'acteurs) ===


dep_name
Bas-Rhin            202
Moselle             200
Gironde             194
Hérault             187
Bouches-du-Rhône    172
Name: nom_amenageur, dtype: int64


=== DÉPARTEMENTS LES MOINS CONCURRENTIELS ===


dep_name
Lozère              18
Wallis et Futuna    14
Guadeloupe          13
Martinique           5
Guyane               2
Name: nom_amenageur, dtype: int64


=== TOP 10 DES DÉPARTEMENTS LES PLUS CONCENTRÉS (Surcharge/Monopole local) ===


,Part du 1er acteur (%),Acteur Dominateur
dep_name,,
Guyane,66.666667,Citeos Cayenne
Lot,61.111111,TE46
Guadeloupe,59.310345,SYNDICAT MIXTE D'ELECTRICITE DE LA GUADELOUPE ...
Lozère,52.657005,Syndicat Départ d'Élec et d'Équip de la Lozère
Deux-Sèvres,51.776650,SEOLIS
Tarn,50.656531,Syndicat Départemental d'Energie du Tarn
Aveyron,49.694190,Syndicat Interco d Énergies du Départ de l Ave...
Haute-Loire,48.632219,406__eborn
Aube,47.788462,SDEA (Syndicat départemental d'énergie de l'Au...


In [86]:
df=pd.read_csv('../data/processed/dataset_irve_enriched.csv')
df.info()

C:\Users\cgboh\AppData\Local\Temp\ipykernel_26212\2303092645.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('../data/processed/dataset_irve_enriched.csv')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166742 entries, 0 to 166741
Data columns (total 29 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id_station_itinerance     166344 non-null  object 
 1   id_pdc_itinerance         166741 non-null  object 
 2   nom_station               166654 non-null  object 
 3   nom_operateur             166316 non-null  object 
 4   nom_amenageur             165355 non-null  object 
 5   adresse_station           166742 non-null  object 
 6   code_insee_commune        166473 non-null  object 
 7   consolidated_code_postal  166742 non-null  object 
 8   consolidated_commune      166742 non-null  object 
 9   puissance_nominale        166742 non-null  float64
 10  implantation_station      166742 non-null  object 
 11  condition_acces           166742 non-null  object 
 12  prise_type_2              166742 non-null  bool   
 13  prise_type_combo_ccs      166742 non-null  b